# Глава 3. Рекуррентные сети

## Введение
Классическое обучение с учителем предполагает выборку независимых одинаково распределённых пар $(x_i, y_i)$ фиксированной размерности. Текстовые данные, однако, нарушают оба допущения: токены внутри последовательности зависимы (порядок имеет значение), а длина переменная . Возникает вопрос, как их эффективно обрабатывать?

Bag-of-words модели, рассмотренные в предыдущей главе, подходят для поиска, но слишком грубы если нужно глубокое понимание текста. Модель не может считать смысл сказанного, невозможно отличить вопрос от утверждения, не говроя о следовании мысли и т.п.

n-грамные языковые модели чуть лучше, но марковское допущение, на основании которых они построены, серьезно ограничивает контекст. Учитывается связь порядка $n-1$. Невозмоно построить эффективную реализацию для n.

Модель Бенджио лучше, поскольку работает уже на уровне семантки. Но главное ограничение, что она обрезает контекст фиксированным окном. То есть можно можно обрабатывать только очень короткие тексты

Выход - рекуррентные сети. Анализируем текст скользящим курсором, а сигнал накапливается по мере прохода по тексту слева напрво. **Рекуррентные нейронные сети** (recurrent neural networks, RNN) [(Elman, 1990)](https://doi.org/10.1207/s15516709cog1402_1) сняли это ограничение: состояние фиксированного размера, обновляемое на каждом шаге, в принципе способно нести информацию о префиксе любой длины. 

<img src="img/rnn.png" width=250>

Рекуррентная языковая модель [(Mikolov et al., 2010)](https://doi.org/10.21437/Interspeech.2010-343) первой уверенно обошла n-граммы, и примерно на десятилетие (2010–2017) RNN стали доминирующей архитектурой NLP

### Типы задач по форме входа и выхода

По соотношению входа и выхода выделяют четыре режима: 
- many-to-one — последовательность в один выход (классификация тональности, детекция спама);
- one-to-many — генерация последовательности из одного входа (порождение текста по затравке, описание изображения);
- синхронный many-to-many — выход на каждом шаге, $T_{вх}=T_{вых}$ (частеречная разметка, NER);
- асинхронный many-to-many — длины входа и выхода различаются (перевод, суммаризация), что потребует отдельной архитектуры encoder–decoder.

Все четыре режима обслуживаются одной и той же рекуррентной ячейкой — меняется только то, где снимается выход.

<img src="img/rnn_output_types.png" width=600>

__Скрытое состояние как память__<br>
Информация о префиксе аккумулируется в векторе $h_t\in\mathbb{R}^{d}$ — скрытом состоянии. Модель делает марковское допущение на уровне состояния: $p(x_{t+1}\mid x_{\le t})\approx p(x_{t+1}\mid h_t)$, то есть $h_t$ — обучаемая достаточная статистика префикса. Это сжатие с потерями: сколь угодно длинная история упаковывается в $d$ чисел. Отсюда и сила RNN (константная память на инференсе), и её главный будущий конфликт — информационное бутылочное горлышко, которое проявится в seq2seq.

Условное ведро, в которое складываете поленую информацию. Рано или поздно ведро переполняется и нужно хранить только самое полезное

## Первые прото сети
Сети дорекуррентной эпохи были реализованы по модели перцептрона: на вход подается некоторый сигнал, он агрегируется как взвешенная сумма и пропускается через функцию активации, генерируя реузультат. Такие модели были как правило однослойными и пасовали перед нелинейными паттернами, что обусловило тупик в развитии искуственного интеллекта. В какой-то момент исследователи задались вопросом: а что если зациклить модель на саму себя: взять выход сети и подать себе же на вход? Так в 80-ые годы появилось и начало активно развиваться направление рекуррентных сетей. 

В 1982 появилась [сеть Хопфилда](https://en.wikipedia.org/wiki/John_Hopfield) - полносвязная однослойная сеть, где каждый нейрон принимает одно из двух значений $s_i\in\{-1,+1\}$, а связь $w_{ij}$ между парой нейронов вычисляется как частоте их совместного срабатывания (в скольки эталонах оба нейрона положительные): $w_{ij}=\frac{1}{N}\sum_{\mu=1}^{P}\xi_i^{\mu}\xi_j^{\mu},\qquad w_{ij}=w_{ji},\quad w_{ii}=0$ $ \, \text{где} \, \xi^{1},\dots,\xi^{P}$. 

Для обучения сети показали эталонные образы, которые нужно запомнить, например, 10 цифр. модель сразу построит карту корреляций (причем сделает это сразу за 1 проход)
Инференс - Модели показывается фрагмент образа или зашумленный образ, модель достраивает до эталона

Сеть моделировала ассоциативная память,
Сеть не первая, были сети Кохонена
Через оптимизацию энергии
Сеть  с обучением по Хэббу
Хэбб - нейробиолог, описываший процесс обучения в мозгу. Согласно модели, если пара нейронов сети срабатывает одновременно, связь между нейронами укрепляется
пример - обучение условным рефлексам, как звонок и еда в экспериментах Павлова
это unsupervised обучение, достаточно показать k эталонов, 
Играем три ноты, вспоминаем мелодию (Shazam)

<img src="img/hopfield1.png" width=200>

Когда на вход подается сигнал, обновления распространяются по сети от нейрона к соседним нейронам. Это происдолит в цикле, каждая такая волна обновления уменьшает целевой функционал - в терминах авторов "энергию системы", пока не упрется в минимум:

$$E = -\frac{1}{2} \sum_{i=1}^{n} \sum_{j=1}^{n} w_{ij} s_i s_j + \sum_{i=1}^{n} \theta_i s_i$$

Модель Хопфилда была детерминриованной и плохо обобщала. В 1985 Хинтон ([Geoffrey Hinton](https://en.wikipedia.org/wiki/Geoffrey_Hinton)) с коллегами модифицировал идею и назвал ее __машина Больцмана__ [(Hinton et al, 1985)](https://www.cs.toronto.edu/~fritz/absps/cogscibm.pdf). Он оставил ту же полносвязную сеть, но сделали скрытыми часть состояний. Это дало модели возможность порождать новые образцы, а не только притягиваться к запомненным. Модель научилась генерировать

Оптимизируемая энергия полной машины Больцмана включает связи всех со всеми - и внутри видимого слоя, и внутри скрытого:

$$E(v,h) = -\sum_i a_i v_i - \sum_j b_j h_j - \sum_{i<i'} u_{ii'} v_i v_{i'} - \sum_{i,j} w_{ij} v_i h_j - \sum_{j<j'} z_{jj'} h_j h_{j'}$$

<img src="img/bm1.png" width=250>

На практике однако машина Больцмана очень медленно сходилась, поэтому разработка имела больше теоретическую ценность, чем практическую. Но уже через год авторы предложили модель __Resticted Bolzman Machine__, где убрали связи между скрытыми нейронами и связи между видимыми нейроноами:

$$E(v,h) = -\sum_i a_i v_i - \sum_j b_j h_j - \sum_{i,j} w_{ij}\, v_i h_j$$

Вычислительный граф стал *двудольным*: при фиксированном $v$ скрытые нейроны условно независимы, поэтому весь слой сэмплируется за один шаг, без обхода соседей:

$$P(h_j = 1 | остальные) = \sigma( b_j + Σ_i w_ij v_i + Σ_{k≠j} w_kj h_k )$$

<img src="img/rbm1.png" width=200>

Модель RBM получила широкое распространение в индустрии и много использовалась, например, была особенно заметной в рекомендательных системах

В 2024 году Джон Хопфилд и Джеффри Хинтон разделили Нобелевскую премию по физике «за основополагающие открытия и изобретения, сделавшие возможным машинное обучение с помощью искусственных нейронных сетей»; в части заслуг Хинтона комитет назвал именно машину Больцмана

Сеть Хопфилда — первый пример реалищации памяти как рекуррентная динамика может служить памятью, адресуемой *по содержанию*: подаёшь искажённый фрагмент образца — получаешь образец целиком. 

А спустя сорок лет [(Ramsauer et al., 2020)](https://arxiv.org/abs/2008.02217) показали, что если разрешить нейронам непрерывные значения и выразить энергию с помощью log-sum-exp, то одно обновление сети Хопфилда записывается как

$$\xi^{\text{new}} = X\,\mathrm{softmax}\big(\beta\, X^{\top}\xi\big),$$

где $X$ — матрица запомненных образцов, а $\xi$ — запрос. Это в точности softmax-внимание, ставшее стандартом и о котором мы поговорим в конце главы: $\xi$ — query, столбцы $X$ — keys и values. То есть ассоциативная память 1982 года и механизм внимания 2014-го оказались одной и той же операцией.

## Модель Элмана
Сложность нейросетевого анализа текстовых данных сводится к двум аспектам: а) нам нужно анализировать большой контекст и б) этот контекст может быть произвольной длины. То есть нужна архитектура, которая сможет универсально работать с любыми текстами. Скользящим окном. Опишем историю появления такой сети

В 1985 году был описан метод обратного распространения ошибки ([error backpropagation](https://en.wikipedia.org/wiki/Backpropagation)) для обучения нейросетей, который обобщил обучение на многослойные сети произвольной глубины и архитектуры. Это дало большой толчок развитию области и в частности рекуррентным сетям, где динамическая глубина вычислений была главной особенностью

В 1986 году Джордан (Michael I. Jordan) описал рекурентную модель для обработки последовательностей произвольной длины. В рамках его модели выход сети на шаге t копировался в специальные контекстные нейроны, а на шаге t + 1 они подавались на вход вместе с очередным элементом последовательности. По сути это была первая рекуррентная сеть в современном понимании, которая могла "запоминать" сигнал и использовать его на протяжении всего вычисления

<img src="img/jordan_rnn.png" width=250>

Джефри Элман (Jeffrey Elman) вдохновился идеей Джордана и предложил небольшую доработку - возвращать на вход не выходы сети, а ее скрытые состояния. Мотивация была в том, что скрытые состояния за счет плотной "упаковки" хранят существенно больше полезной информации, чем выходной слой. Так появилась сеть Элмана:

$$\begin{cases}h_t=f\big(W_{xh}\,x_t+W_{hh}\,h_{t-1}+b_h\big),\qquad \\ y_t=g\big(W_{hy}\,h_t+b_y\big),\end{cases}$$

где $x_t\in\mathbb{R}^{e}$ — вход (например, эмбеддинг токена)<Br>$W_{xh}\in\mathbb{R}^{d\times e}$, $W_{hh}\in\mathbb{R}^{d\times d}$ - обучаемые матрицы преобразования<br>$f$ — функция активации для состояния (как правило tanh)<br>$g$ - функция активации для выхода: softmax для языковой модели, сигмоида для бинарной классификации

Данная архитектура стала фактическим стандартом и по сей день на нее ссылаются, когда говорят о рекуррентных сети.

<img src="img/elman_rnn.jpeg" width=300>

Любое рекуррентное выражение вида $x = f(x)$ можно полностью развернуть до своего начала $f(0)$, выписав его одним выражением, для рекуррентной ети это называют __Unfolding__. Так, если обрабатывается последовательность из T токенов, в развернутом варианте это будет единый граф вычислений глубины $T$ с большим кол-вом повторяющихся весов

<img src="img/unfold.png" width=300>

Видна хараткерная особенность. Многократное повторение умножения на одну и ту же матрицу обнажает одновременно<br> 
а) высокую выразительность внутренних представлений - у нас много нелинейных активаций<br>
б) большое кол-во проблем и сложностей обучения, главным образом, затухание и экспоненциальный взрыв градиентов

А качестве функции активации обычно используют гиперболический тангенс tanh(). может ReLU()

Чем инициализировать начальное состояние
- $h_0=0$ — стандарт: до начала текста контекста нет
- обучаемый $h_0$ полезен на коротких последовательностях, где старт вносит заметный вклад: выучивается априорный контекст
- cлучайный шум в $h_0$ при обучении — лёгкая регуляризация, снижающая зависимость от начала
- $h_0$ очередного чанка равен $h_T$ предыдущего (stateful-режим) — используется при обучении на длинных потоках усечённым BPTT (см. ниже)

## Обучение модели Элмана
Главное новаторство Элмана было в том, что он описал рекуррентную архитектуру. Но дело в том, что обучал он свою модель примитивным способом

Полноценный процесс обучения, каким мы его знаем сегодня, описал другой математик Пол Вербос, который известен тем, что ранее сформулировал метод обратного распространения ошибки. метод был хорошо известен, но для RNN не подходил, поскольку рекрурентость соотношений добавляет специфику. Да, сеть разворачивается во вполне стандартный многослойный граф вычислений, но в отличии от обычного MLP, все слои в этом графе общие (shared), то есть формулы должны были это учесть. Так появился метод обратного распространния ошибки по времни **BPTT** (backpropagation through time) [(Werbos, 1990)](https://doi.org/10.1109/5.58337)

Рассмотрим развернутый граф $$
y = f\Big( W \cdot f\big( W \cdot f(W \cdot h_0) \big) \Big)
$$

По формуле полной проивзодной суммируются частные производные


Для суммарных потерь $L=\sum_t L_t$ вклад шага $t$ в градиент по $W_{hh}$ собирается со всех предшествующих позиций $k\le t$:

$$\frac{\partial L_t}{\partial W_{hh}}=\sum_{k=1}^{t}\frac{\partial L_t}{\partial h_t}\left(\prod_{i=k+1}^{t}\frac{\partial h_i}{\partial h_{i-1}}\right)\frac{\partial^{+} h_k}{\partial W_{hh}},$$

где $\partial^{+}h_k/\partial W_{hh}$ — «немедленная» производная при замороженном $h_{k-1}$. Вся динамика обучения спрятана в произведении якобианов соседних шагов

В отличие от обычной глубокой, в рекуррентной сети матрица одна и та же на всех шагах ($W_{hh}$) => при подсчете градиента  суммировать

## Проблемы рекуррентной модели

#### Проблема 1: сжимающий процесса
Начнем для простоты с ситуации, когда функция активации нет, тогда можем сразу выписать выражение для любого шага $h_t$ как функцию от начального состояния $h_0$:

$$h_t = W_{hh}^t h_0 + \sum_{k=1}^{t} W_{hh}^{t-k} W_{xh} x_k$$

Оценим вклад первого слагаемого. По свойству произведения матриц:

$$\|W_{hh}^t h_0\| \le \|W_{hh}\|^t \cdot \|h_0\|$$

Если матрица $W_{hh}$ реализует "сжимающее" отображение, то есть ее норма меньше единицы ( $\|W_{hh}\| = \gamma < 1$), то ее степень $W_{hh}^t$ стремится к нулю при больших $t$. Значит и верхняя граница всего слагаемого $\|W_{hh}^t h_0\| \le \gamma^t \|h_0\|$ неизбежно стремится к нулю. Это доказывает, что вклад начального состояния $h_0$ экспоненциально стирается из скрытого состояния $h_t$

Если есть нелинейная функция активации $h_t = \tanh(W_{hh} h_{t-1} + W_{xh} x_t)$ - вытащить $h_0$ в виде отдельного слагаемого нельзя. Поэтому вместо анализа вклада изолированного слагаемого мы исследуем расстояние между двумя траекториями: запустим одну и ту же сеть с двумя разными начальными состояниями $h_0$ и $\hat{h}_0$ на одинаковой последовательности входов.

Связь между состоянием траекторий описывается неравенством через константу Липшица:

$$\|h_t(h_0) - h_t(\hat{h}_0)\| \le \left( L_{\sigma} \cdot \|W_{hh}\| \right)^t \cdot \|h_0 - \hat{h}_0\|$$

Здесь $L_{\sigma}$ — константа Липшица функции активации, она детерминирована, напеример, для $\tanh$ она равна $1$. Выражение $\|h_t(h_0) - h_t(\hat{h}_0)\|$ измеряет разность между состояниями на шаге $t$.

В этом неравенстве к нулю стремится компонент $(L_{\sigma} \cdot \|W_{hh}\|)^t$: так как $L_{\sigma} \le 1$, а для сжимающей матрицы $\|W_{hh}\| < 1$, их произведение $\gamma = L_{\sigma} \|W_{hh}\|$ строго меньше единицы, и при $t \to \infty$ получаем $\lim_{t \to \infty} \gamma^t = 0$.

Поскольку компонент $\gamma^t \to 0$, вся правая часть неравенства стремится к нулю. Это доказывает, что траектории с разными начальными состояниями экспоненциально сходятся к одной, и сеть полностью теряет чувствительность к начальным условиям

<img src="img/rnn_trajectories.png" width=500>

### Проблема 2: расширяющийся процесс
Рассмотрим проитвоположный случай, когда спектральный радиус матрицы весов больше единицы: $\rho(W_{hh}) > 1$

В отсутствие функции активации:

$$h_t = W_{hh}^t h_0 + \sum_{k=1}^{t} W_{hh}^{t-k} W_{xh} x_k$$

Первое слагаемое в бесконечность.

Если применяется нелинейная функция активации:

$$h_t = \tanh(z_t), \quad \text{где } z_t = W_{hh} h_{t-1} + W_{xh} x_t + b_h$$

Выпишем норму преактивации $z_t$ и сравним два ее компоненита через треугольное неравенство:

$$\|z_t\| = \|W_{hh} h_{t-1} + (W_{xh} x_t + b_h)\| \ge \|W_{hh} h_{t-1}\| - \|W_{xh} x_t + b_h\|$$

Используя свойства собственного значения $|\lambda_1| > 1$, получаем:

$$\|z_t\| \ge |\lambda_1| \cdot \|h_{t-1}\| - C_{input}$$

где $C_{input} = \|W_{xh} x_t + b_h\|$ — ограниченная величина входного сигнала.

Поскольку $|\lambda_1| > 1$, при каждом умножении на $W_{hh}$ норма вектора аргумента $z_t$ увеличивается. Если $\|h_{t-1}\|$ превышает порог $\frac{C_{input}}{|\lambda_1| - 1}$, то $\|z_t\| > \|h_{t-1}\|$, и компоненты вектора $z_t$ начинают монотонно расти по модулю с каждым временным шагом:

$$\lim_{t \to \infty} |z_{t, i}| = \infty \quad \text{для компонентов } i$$

$$\lim_{z \to +\infty} \tanh(z) = 1, \quad \lim_{z \to -\infty} \tanh(z) = -1$$

Таким образом, монотонный рост аргумента $z_t$ под действием расширяющего оператора $W_{hh}$ приводит к тому, что значения $h_t = \tanh(z_t)$ зажимаются на границах области значений $\pm 1$ (область глубокого насыщения).

При расширяющей матрице $W_{hh}$ компоненты вектора скрытого состояния $h_t$ быстро увеличиваются по модулю, заставляя нелинейную функцию активации $\sigma(z) = \tanh(z)$ входить в область глубокого насыщения ($h_t \approx \pm 1$).

<img src="img/rnn_expanding.png" width=500>

Покажем что вход x не влияет на сигнал. Чувствительность текущего скрытого состояния $h_t$ к новому входному сигналу $x_t$:

$$\frac{\partial h_t}{\partial x_t} = \operatorname{diag}(1 - h_t^2) W_{xh}$$

Когда состояние $h_t$ находится в области насыщения, диагональные элементы матрицы производных $\sigma'(z_t) = 1 - h_t^2$ стремятся к нулю:

Поскольку компонент $(1 - h_t^2) \to 0$, вся матрица частных производных стремится к нулевой. Это доказывает, что при расширяющей матрице $W_{hh}$ новые входы $x_t$ попадают на плоские участки функции активации и перестают оказывать влияние на скрытое состояние сети.

### Проблема 3: Затухание градиента

При обучении сети методом BPTT градиент функции потерь $L_T$ по скрытому состоянию на отдаленном шаге $t \ll T$ вычисляется по Chain Rule через произведение Якобианов перехода:

$$\frac{\partial L_T}{\partial h_t} = \frac{\partial L_T}{\partial h_T} \prod_{k=t+1}^{T} J_k, \quad \text{где } J_k = \operatorname{diag}(\sigma'(z_k)) W_{hh}^T$$

Оценим норму этого градиентного сигнала с помощью свойств матричных норм:

$$\left\| \frac{\partial L_T}{\partial h_t} \right\| \le \left\| \frac{\partial L_T}{\partial h_T} \right\| \cdot \prod_{k=t+1}^{T} \|J_k\| \le \left\| \frac{\partial L_T}{\partial h_T} \right\| \cdot \left( L_{\sigma} \cdot \|W_{hh}\| \right)^{T-t}$$

Верхняя граница нормы градиента стремится к нулю. Это доказывает, что сигнал ошибки не доходит до ранних шагов последовательности, ограничив горизонт обучения сети коротким интервалом в 5–10 шагов.

### Проблема 4: Взрыв градиента

Рассмотрим противоположную ситуацию при BPTT, когда норма Якобианов перехода строго больше единицы: $\|J_k\| = \Gamma > 1$.

Оценим снизу норму произведения Якобианов при передаче градиентного сигнала на расстояние $T - t$:

$$\left\| \prod_{k=t+1}^{T} J_k \right\| \sim \Gamma^{T-t}$$

В этом выражении к бесконечности стремится компонент $\Gamma^{T-t}$: при $T - t \gg 1$ возведение числа больше единицы в степень длины интервала дает $\lim_{(T-t) \to \infty} \Gamma^{T-t} = \infty$.

Поскольку компонент $\Gamma^{T-t} \to \infty$, величина градиентного шага $\Delta W_{hh} \propto -\eta \frac{\partial L_T}{\partial W_{hh}}$ неограниченно растет. Это доказывает, что при расширяющих операторах обратный проход приводит к численной переполняемости (ошибки NaN/Inf) и гигантским скачкам весов, которые разрушают процесс оптимизации

Чтобы строго доказать, почему норма вектора скрытого состояния $h_t$ определяется наибольшим по модулю собственным значением (спектральным радиусом $\rho(W_{hh})$), независимо от наличия сжимающих осей, воспользуемся спектральным разложением.

Для оценки максимального возможного растяжения любого вектора единичной нормы используется **Формула Гельфанда** (Gelfand's formula), связывающая матричную норму со спектральным радиусом:

$$\rho(W_{hh}) = \lim_{t \to \infty} \|W_{hh}^t\|^{1/t}$$

Это означает, что для любого $\varepsilon > 0$ при достаточно больших $t$ норма оператора зажата в границах:

$$\left( \rho(W_{hh}) - \varepsilon \right)^t \le \|W_{hh}^t\| \le \left( \rho(W_{hh}) + \varepsilon \right)^t$$

Бенжио с коллегами исследовал возможности [(Bengio et al., 1994)](https://doi.org/10.1109/72.279181). Авторы показали, что RNN может надёжно выучивать зависимости длиной не более 5–20 шагов

В работе [(Pascanu et al., 2013)](https://arxiv.org/abs/1211.5063) дали более глубокое описание фундаментальных ограничений базовой RNN модели Элмана. Они рассмотрели проблему сразу с трех позиций:
в дополнение к математике собственных значений с позиции динамической системы, а также геометрическойю Там же они предложили два практических инструмента: "gradient clipping" для  борьбы с взрывающимся градиентом и регуляризация для борьбы с vanishing gradient

## Методы стабилизации обучения

__Gradient clipping__<br>
Gradient clipping = искусственное ограничение слишком большого градиента, возникающего в процессе обучения:
- по норме<br>если градиент слишком большой $\lVert g\rVert>\theta$, то он $g\leftarrow\theta\,g/\lVert g\rVert$ — нормируется, типичные $\theta\in[1,5]$<br><br>
- по отдельным параметрам<br>$g_i\leftarrow\operatorname{clip}(g_i,-\theta,\theta)$ — так проще, но искажает направление градиента

__Регуляризация__<br>

$L_2$-регуляризация — стандартный инструмент обучения нейросетей, но в рекуррентных сетях с ней надо осторожнее: она сжимает $W_{hh}$, уменьшая спектральный радиус, и тем самым усугубляет затухание градиента. ✏️ *(В исходном тексте фраза была оборвана: «для регуляризации используют , но...».)* На практике рекуррентные веса $W_{hh}$ либо исключают из weight decay, либо берут для них коэффициент на порядок меньше, а полновесный $L_2$ применяют к входным и выходным матрицам. $L_1$ (разреживание) на рекуррентных весах используется редко.

Добавляем в функцию потерь штраф за слишком большое или слишком маленькое изменение градиента

$$
\Omega = \sum_{t} \left( \frac{ \left\| \frac{\partial L}{\partial \mathbf{x}_{t+1}} \cdot \frac{\partial \mathbf{x}_{t+1}}{\partial \mathbf{x}_t} \right\| }{ \left\| \frac{\partial L}{\partial \mathbf{x}_{t+1}} \right\| } - 1 \right)^2
$$

Дробь отражает отношение градиента на шаге t к градиенту на шаге t+1. Просуммировано по всем расстояниям

**Dropout**

Напомним, что Dropout - это зануление случайных весов модели во время обучения. По сути искусственное усложнение, для получения более обобщающей модели

Наивный dropout в рекуррентной среде работает плохо. Если мы генерируем новую маску на каждом шаге, сигнал быстро деградирует по мере продвижения по последовательности. Потому что матожидание активаций сохраняется, а дисперсия накапливается мультипликативно, и на длинных последовательностях состояние превращается в шум

Чтобы обойти ограничение, можно оставить dropout только на нерекуррентных связях [(Zaremba et al., 2014)](https://arxiv.org/abs/1409.2329). Можно сгенерировать маску один раз перед прогоном модели и применяется на всех шагах, как в подходе **Вариационный dropout** [(Gal & Ghahramani, 2016)](https://arxiv.org/abs/1512.05287). Либо следовать методу **Zoneout** [(Krueger et al., 2016)](https://arxiv.org/abs/1606.01305) — с вероятностью $p$ компонента состояния не пересчитывается, а копируется с предыдущего шага по времени:
$$h_t=m_t\odot h_{t-1}+(1-m_t)\odot\tilde h_t,\qquad m_t\sim\operatorname{Bernoulli}(p)$$

**Layer Normalization**

Нормализация в нейросетях это приведение весов к некоторой ограниченной шкале через вычистаение среднего и деления на дисперсию. Для настройки отображения нужно собрать статистику по сигналу и часто для сбора статистики используют текущий батч с данными, на которых обучается модель

В рекуррентной среде Batch normalization работает плохо, так как последовательности в батче разной длины. Для больших t статистики считаются по горстке примеров. Вместо него предпочитают LayerNorm - нормализацию по измеренияям одного примера [(Ba et al., 2016)](https://arxiv.org/abs/1607.06450). Обычно нормализацию ставят на предактивации внутри ячейки, отдельно на вход и на рекуррентную часть

$$\operatorname{LN}(a)=g\odot\frac{a-\mu}{\sqrt{\sigma^{2}+\varepsilon}}+b,\qquad \mu=\tfrac1d\sum_i a_i,\ \ \sigma^2=\tfrac1d\sum_i (a_i-\mu)^2$$

**Ортогональная инициализация** 

Инициализация весов нейронной сети важный шаг обучения. В рекруррентной постановке имеет особое значение. 

Вместо инициализации матрицы $W_{hh}$ случайным шумом [(Saxe et al., 2013)](https://arxiv.org/abs/1312.6120) предложили брать какую-либо ортогональную матрицу  

Напомним, что матрица - ортогональная, если все её стоблцы - ортонормированные векторы (ортгональны друг другу и единичной нормы). Ортогональность матрицы эквивалентна равенству 1 всех ее сингулярных значений. А как мы выяснили выше, сингулярное число определяет стабильность процесса накполнения сигнала. Одна из интерепретаций ортогональная матрица - это всегда матрица повторота / отражения, она не удлиняет и не укорчивает

Как можно получить ортогональную матрицу:
- QR разложение случайной Гауссовой матрицы, взять матрицу Q<br><br>*В алгебре QR разложение - представление матрицы в виде произведения ортогональной Q и врехнетреугольной R (фактически это визуализация процесса [ортогонализации Грамма-Шмидта](https://ru.wikipedia.org/wiki/%D0%9F%D1%80%D0%BE%D1%86%D0%B5%D1%81%D1%81_%D0%93%D1%80%D0%B0%D0%BC%D0%B0_%E2%80%95_%D0%A8%D0%BC%D0%B8%D0%B4%D1%82%D0%B0))<br><br>
- в SVD разложении матрицы U и V тоже ортогональны

PS здесь именно про инициализацию, чтобы поддерживать в течении всего времени обучения, есть подходы основанные на регуляризации

**Truncated BPTT** [(Williams & Peng, 1990)](https://doi.org/10.1162/neco.1990.2.4.490)<br>длинный поток режется на чанки по $k$ шагов; состояние переносится между чанками вперёд, а backpropagation идёт только внутри чанка. Память требует места уже не под $O(Td)$ а под $O(kd)$ активаций, обновления учащаются. Цена — смещение градиента: через границы чанков он не течёт, и зависимости длиннее $k$ напрямую не обучаются, лишь косвенно через перенесённое состояние. Общая форма TBPTT($k_1$, $k_2$) — обновление каждые $k_1$ шагов с разворачиванием на $k_2$ назад

RNN можно использовать и для генерация — цикл: предсказанный $\hat y_t$ подаётся на вход шага $t+1$ (жадный argmax, сэмплирование или beam search — детали в кейсах). Состояние обновляется на месте, поэтому RNN генерирует каждый следующий токен за $O(1)$ памяти и вычислений независимо от длины уже сгенерированного. Этого свойства будут лишены трансформеры (KV-кэш растёт линейно), и его же вернут state-space модели.

## Модель LSTM
В 1991 году Зепп Хохрайтер (Sepp Hochreiter) в своей дипломной работе описал проблемы, связанные с многократным перемножением сигнала в, из-за чего процесс фундаментально неустойчив с ростом длины входной последовательности. Вместе со Шмидхубером он предложил принципиальную модификацию рекуррентной модели Элмана — **LSTM**, Long Short-Term Memory [(Hochreiter & Schmidhuber, 1997)](https://doi.org/10.1162/neco.1997.9.8.1735)

Шаг 1 - убрать мульиипликативность памяти. Помимо привычного скрытого состояния LSTM передаёт отдельный сигнал долговременной памяти — cell state $c_t$, — который обновляется не умножением, а сложением:

$$c_t = c_{t-1} + i_t\odot\tilde c_t$$

Отсюда $\partial c_t/\partial c_{t-1}=I$ — **карусель постоянной ошибки** (constant error carousel): вдоль этой траектории градиент проходит сотни шагов, не затухая, потому что на ней вообще нет матричного умножения. Сравните с $\operatorname{diag}(f')W_{hh}$ у Элмана — именно это произведение мы только что разбирали в разделе про устойчивость. Скрытое состояние $h_t$ при этом становится рабочей *проекцией* памяти, а не самой памятью.

Шаг 2 - вентили (gates). Главная техническая новация. Модель сама принимает решение, что и когда запоминать. Инструмент — сигмоидные векторы из $(0,1)^d$, поэлементно умножающие информационные потоки: 0 — закрыто, 1 — открыто, между — частично. Степень пропускания вычисляется из текущего контекста $[h_{t-1};x_t]$, то есть сеть решает это сама и по-разному на каждом шаге

Шаг 3 - гейт забывания. В версии 1997 года его не было — его добавили тремя годами позже [(Gers et al., 2000)](https://doi.org/10.1162/089976600300015015), когда выяснилось, что на непрерывных потоках ячейка без механизма очистки дрейфует и насыщается. $f_t$ поэлементно масштабирует старую память: конец предложения — обнулить синтаксический контекст, смена темы — стереть тематический. Градиент магистрали становится $\partial c_t/\partial c_{t-1}=\operatorname{diag}(f_t)$: скоростью «утечки» памяти управляет сама сеть и может держать её сколь угодно близко к единице

**Полная система**

$$\begin{cases}
f_t&=\sigma(W_f[h_{t-1};x_t]+b_f) && \text{гейт забывания: что стереть из } c\\
i_t&=\sigma(W_i[h_{t-1};x_t]+b_i) && \text{гейт входа: насколько вписать кандидата}\\
\tilde c_t&=\tanh(W_c[h_{t-1};x_t]+b_c) && \text{кандидат: что именно вписать}\\
c_t&=f_t\odot c_{t-1}+i_t\odot\tilde c_t && \text{обновление памяти}\\
o_t&=\sigma(W_o[h_{t-1};x_t]+b_o) && \text{гейт выхода: что показать наружу}\\
h_t&=o_t\odot\tanh(c_t) && \text{выход}
\end{cases}$$

<img src="img/lstm.png" width=350>

__Инициализация__<br>
Из-за многократного умножения матрицы весов, корректная инциализация матриц крайне важна для RNN моделей. Отдельный вопрос, чем инициализировать смещение $b_f$. При инициализации $f_t\approx 0.5$  забывание как $0.5^{T}$ ещё до начала обучения, и градиентный сигнал о пользе долгой памяти не успевает дойти до весов.

Для борьбы с этим эффектом достаточно поставить bias побольше. нарипмер можно использовать детерминированный bias: $b_f=1\ldots 2$, тогда средняя активация будет начинаться с $\sigma(1)\approx 0.73$, то есть режим по умолчанию "помнить", забыванию — учиться

В 2015 году Sustekever с коллегами из Google провели масштабное сравнение архитектур рекуррентой сети: эволюционным поиском они перебрали более десяти тысяч вариантов архитектуры рекуррентных ячеек [(Jozefowicz et al., 2015)](https://proceedings.mlr.press/v37/jozefowicz15.html). В том числе они:
- подтвердили рекомендацию прибавлять единицу во время инициализации гейта забывания $b_f$ — это самая дешёвая правка с самым заметным эффектом
- проранжировали гейты по важности: Forget, Input, Output
- LSTM и GRU оказались практически неотличимы по качеству и уверенно лучше простой RNN; несколько найденных вариантов (MUT1–MUT3) шли с GRU вровень, но систематического преимущества ни одна не дала, что само по себе сильный аргумент в пользу того, что дело не в конкретной формуле гейтинга, а в самом факте аддитивной памяти

**Peephole-соединения** [(Gers & Schmidhuber, 2000)](https://doi.org/10.1109/IJCNN.2000.861302)<br>В 2000 году те же авторы предложили небольшую модификацию своей модели - разрешиили при вычислении каждого гейта модель «подглядывать» в текущее состояние.

Модифицированые формулы выглядят так:

$$
\begin{cases}
f_t &= \sigma\!\left(W_f\,[h_{t-1}, x_t] + \textcolor{blue}{w_{cf} \odot c_{t-1}} + b_f\right) && \text{гейты забывания} \\
i_t &= \sigma\!\left(W_i\,[h_{t-1}, x_t] + \textcolor{blue}{w_{ci} \odot c_{t-1}} + b_i\right) && \text{гейты входа} \\
\tilde{c}_t &= \tanh\!\left(W_c\,[h_{t-1}, x_t] + b_c\right) && \text{состояние} \\
c_t &= f_t \odot c_{t-1} + i_t \odot \tilde{c}_t && \text{гейты забывания} \\
o_t &= \sigma\!\left(W_o\,[h_{t-1}, x_t] + \textcolor{blue}{w_{co} \odot c_t} + b_o\right) && \text{гейты выхода} \\
h_t &= o_t \odot \tanh(c_t) && \text{выход}
\end{cases}
$$

По задумке эта модификация активизирует у модели более точное "ощущение времени": гейты могут видять накопленные счётчики напрямую, минуя фильтр $o_t$. Это помогает в задачах, где требуется точный счёт. Однако на большинстве NLP задач систематического выигрыша не даёт и частью стандартной реализации так и не стало.

<img src="img/lstm_peephole.webp" width=350>

## Модель GRU
В 2014 году исследовательская группа [(Cho et al., 2014)](https://arxiv.org/abs/1406.1078) представила модель **GRU** = gated recurrent unit . Она реализовывала тот же принцип что LSTM, но упрощала ее избыточно сложную и нагруженную архитектуру. Идея была такой: зачем нам 2 отдельных гейта для контроля записи (Input + Forget), когда это можно релазовать одним. В итоге общее кол-во гейтов сократили с трех до двух: Update + Reset

$$\begin{cases}
z_t&=\sigma(W_z[h_{t-1};x_t]) && \text{гейты обновления} \\
r_t&=\sigma(W_r[h_{t-1};x_t]) && \text{гейты выхода} \\
\tilde h_t&=\tanh(W_h[r_t\odot h_{t-1};x_t]) && \text{выход}\\
h_t&=z_t\odot h_{t-1}+(1-z_t)\odot\tilde h_t && \text{гейты забывания}
\end{cases}$$

Гейты обновления $z_t$ выполняют работу пары forget+input со встроенной связью $f=1-i$: новое состояние — выпуклая комбинация старого и кандидата. При $z_t\to 1$ прошлое копируется, при $z_t\to 0$ замещается новым. «Раздуться», как $c_t$ у LSTM, оно не может; выходных ворот нет: $h_t$ — одновременно и память, и выход

$r_t$ действует до вычисления кандидата: $\tilde h_t$ может «не видеть» нерелевантное прошлое и начать с чистого листа, при этом само состояние ещё не стёрто — сотрёт его или нет, решит $z_t$. Разделение труда: $r$ — краткосрочная релевантность контекста, $z$ — долгосрочный баланс старого и нового

Сравнение LSTM и GRU по кол-ву праметров

Тесты [(Chung et al., 2014)](https://arxiv.org/abs/1412.3555) показали что гейтинг важен для RNN моделей, а GRU и LSTM идут практически вровень. При этом инженерно GRU проще

Поиск по пространству вариантов [(Greff et al., 2015)](https://arxiv.org/abs/1503.04069): критичны гейты забывания и выходная нелинейность, peephole и прочие вариации погоды не делают. Эвристика: мало данных или жёсткий бюджет — GRU (меньше параметров — меньше переобучение, быстрее); большие корпуса, языковое моделирование и перевод — LSTM (чуть выше потолок качества); универсальный дефолт — LSTM с $b_f=1$.

## Масштабирование архитектуры
У классичексой RNN всего 1 слой (если считаем уникальные веса и не "разворачиваем" ее). Почему бы не сделать сеть многослойной. Так появились __Stacked RNNs__. Выходная последовательность слоя $l-1$ служит входной для слоя $l$, вместо исходного текста орбабатываем промежуточные представления

$$h_t^{(l)}=\operatorname{Cell}^{(l)}\big(h_t^{(l-1)},\,h_{t-1}^{(l)}\big)$$

Cоздается иерархия признаков. Нижние слои отвечают за детекцию низкоуровенвых паттернов (орфографию и морфологию), верхние — за синтаксис и семантику. 

<img src="img/stacked_rnn.svg" width=250>

Глубокие рекуррентные стеки впервые убедительно выстрелили в распознавании речи [(Graves et al., 2013)](https://arxiv.org/abs/1303.5778); Для NLP типично использовать 2-4 слоя, на более глубоких сетях без остаточных связей (residual connections) качество не растёт.

**Bi-RNN** [(Schuster & Paliwal, 1997)](https://doi.org/10.1109/78.650093)<br>Обрабатывать входную последовательность можно как слева-направо так и справа-налево. Ничто не мешает это делать одновременно, а сигналы двух проходов объединить. Так появились двунаправленные RNN. 

Bi-RNN = две независимые сети, которые читают последовательность слева направо и справа налево, а состояния комбинируются конкатенацией $h_t=[\overrightarrow{h}_t;\overleftarrow{h}_t]$, а для классификации всей последовательности — $[\overrightarrow{h}_T;\overleftarrow{h}_1]$. Мотивация: у слова есть и левый, и правый контекст (омонимия часто разрешается словами справа). Незаменима для задач Natural Understanding (энкодеры перевода), а также задач разметки, NER

<img src="img/birnn.svg" width=250><br><br>

**BiLSTM-CRF** [(Huang et al., 2015)](https://arxiv.org/abs/1508.01991), [(Lample et al., 2016)](https://arxiv.org/abs/1603.01360)

[уточнить]

Каноническая архитектура разметки последовательностей той эпохи, и её стоит разобрать, потому что она вскрывает ограничение всех рассмотренных моделей. BiLSTM выдаёт на каждой позиции распределение по тегам *независимо* от соседних позиций. Но теги зависимы: в схеме BIO переход `O → I-PER` запрещён грамматикой разметки, а не статистикой. Независимый argmax по позициям такие переходы порождает регулярно.

Решение — надстроить над выходами BiLSTM линейно-цепной CRF, который добавляет обучаемую матрицу переходов $A_{ij}$ и оценивает последовательность тегов целиком:

$$s(x,y)=\sum_{t} \underbrace{P_{t,y_t}}_{\text{от BiLSTM}} + \sum_{t} \underbrace{A_{y_{t-1},y_t}}_{\text{переходы}},\qquad p(y\mid x)=\frac{e^{s(x,y)}}{\sum_{y'}e^{s(x,y')}}$$

Знаменатель считается вперёд-алгоритмом за $O(T|\mathcal{Y}|^2)$, декодирование — Витерби. Прибавка к F1 на NER обычно 1–3 пункта, и почти вся она — за счёт исчезновения запрещённых переходов. Идея живёт и сегодня: CRF-голову регулярно ставят поверх BERT.

**ELMo** [(Peters et al., 2018)](https://arxiv.org/abs/1802.05365)

Важное развитие подхода — предобученная многослойная biLSTM языковая модель как источник контекстных эмбеддингов. Одна из первых текстовых моделей, обученных в парадигме transfer learning: сначала универсальный pretrain на большом корпусе, затем адаптация под узкую задачу.

Отличие - ELMo отдаёт не выход верхнего слоя, а обучаемую взвешенную сумму всех слоёв: $\mathrm{ELMo}_k=\gamma\sum_{l} s_l\,h_{k}^{(l)}$, где веса $s_l$ настраиваются под конкретную задачу. Если задача синтаксическая, больше синала содержится в нижних слоях, если семантическая, то верхние

Название запустило традицию именовать NLU модели героями «Улицы Сезам»: сначала вышла ELMo, затем появился BERT (о нем подробнее в главе про трансформеры), дальше пошли ERNIE, Grover и прочее

<img src="img/elmo1.png" width=500>

Обратной сети нужен конец последовательности для начала работы, поэтому двунаправленность применима не во всех задачах: онлайн-распознавание речи, синхронный перевод, автодополнение. Там либо только однонаправленные модели, либо компромиссы вроде блочной двунаправленности.

**GNMT** [(Wu et al., 2016)](https://arxiv.org/abs/1609.08144)<br>
Многослойные RNN подвержены тем же проблемам, что и обычные глубокие сети. И решение то же - использование остаточных связей (__residual connection__) $h^{(l)}=h^{(l-1)}+\operatorname{Cell}^{(l)}(\cdot)$. Образцовый пример - модель GNMT, реализующая 8-слойный LSTM-стек с residual-связями

<img src="img/gnmt.png" width=250>

__AWD-LSTM__

В 2017 году выпустили LSTM на стреоидах: ASGD Weight-Dropped LSTM. Рекуррентная трехслойная сеть.

Центральный приём, давший название модели, — DropConnect на матрицах скрытых состояний: вместо зануления активаций случайно зануляются сами веса рекуррентных связей, причём маска генерируется один раз на прямой проход, а не на каждом шаге времени, благодаря чему приём совместим с оптимизированными библиотечными реализациями LSTM и не замедляет обучение. 

Вторая часть названия относится к NT-ASGD — усреднённому стохастическому градиентному спуску с недетерминированным триггером: момент переключения на усреднение весов определяется автоматически, когда качество на валидации перестаёт улучшаться заданное число проверок подряд, что избавляет от подбора этого момента вручную. 

Дополнительно применяются вариационный дропаут (одна и та же маска на всех временных шагах последовательности), дропаут на уровне слов в матрице эмбеддингов, связывание весов эмбеддингов и выходного softmax-слоя, обратное распространение по последовательностям переменной длины с пропорциональной коррекцией шага обучения, а также регуляризация активаций (AR) и их временных изменений (TAR) — штраф на норму скрытых состояний и на норму разности между соседними по времени состояниями

__xLSTM__
xLSTM = Extended LSTM (Beck et al, 2024). Работа отвечает на вопрос, насколько далеко можно продвинуться в языковом моделировании, если масштабировать LSTM до миллиардов параметров, применив накопленные при разработке больших языковых моделей приёмы и одновременно устранив три известных ограничения рекуррентных сетей: невозможность пересмотреть однажды принятое решение о записи в память, ограниченную ёмкость памяти, сжатой в скалярное состояние ячейки, и отсутствие параллелизуемости из-за смешивания памяти — рекуррентных связей между скрытыми состояниями соседних шагов. 

Первое решение — экспоненциальное стробирование: сигмоида во входном и забывающем вентилях заменяется экспонентой, что позволяет резко переоценивать важность новой информации относительно уже сохранённой; поскольку экспонента способна вызвать переполнение, вводятся дополнительное состояние-нормализатор и стабилизирующее состояние, переводящее вычисления в логарифмическую шкалу. 
Второе решение — два новых типа ячеек. sLSTM сохраняет скалярную память и скалярное обновление, но получает несколько «голов» со смешиванием памяти внутри головы и без смешивания между головами, что даёт новый механизм взаимодействия ячеек; такая ячейка остаётся строго рекуррентной и непараллелизуемой, поэтому для неё написаны специализированные CUDA-ядра. mLSTM заменяет скалярное состояние матрицей размера d × d, обновляемой по ковариационному правилу — внешним произведением векторов ключа и значения, как в ассоциативной памяти; извлечение выполняется умножением матрицы памяти на вектор запроса. Отказ от смешивания памяти лишает mLSTM рекуррентных связей по скрытому состоянию, благодаря чему рекуррентность допускает эквивалентную параллельную формулировку и обучение идёт так же, как у трансформера, тогда как при генерации сохраняются линейная по длине последовательности сложность и постоянный расход памяти. Ячейки помещаются в остаточные блоки двух видов — с проекцией размерности после операции (по образцу трансформера, обычно с sLSTM) и до неё (по образцу моделей пространства состояний, обычно с mLSTM), — а блоки стекируются в итоговую архитектуру; их соотношение записывается в обозначении вида xLSTM[7:1]. По качеству и характеру масштабирования xLSTM оказывается сопоставим с современными трансформерами и моделями пространства состояний, демонстрируя при этом хорошую экстраполяцию на длины контекста, превышающие использованные при обучении. Позднее авторы обучили модель xLSTM 7B на 2,3 трлн токенов, подтвердив работоспособность подхода на масштабе больших языковых моделей.

## Encoder–Decoder (Seq2Seq)

**Seq2Seq** (encoder–decoder) [(Cho et al., 2014)](https://arxiv.org/abs/1406.1078), [(Sutskever et al., 2014)](https://arxiv.org/abs/1409.3215): энкодер читает $x_{1..S}$ и сжимает её в контекстный вектор $v$ (обычно последнее состояние $h_S$); декодер — условная языковая модель $p(y_t\mid y_{<t},v)$, инициализированная $v$. Обучение — суммарная кросс-энтропия по цепному правилу. Инженерные находки Sutskever: четырёхслойные LSTM, разворот входной последовательности (первые слова источника оказываются рядом с первыми словами перевода — критические зависимости короче) и beam search при декодировании

<img src="img/seq2seq.svg" width=450>

Предложения протискивается через один вектор фиксированной размерности - он является "бутылочным горлышком". Качество перевода заметно деградирует с ростом длины фразы, исследование  [(Cho et al., 2014b)](https://arxiv.org/abs/1409.1259) это наглядно проиллюстрировало. 

Наращивание размерности вектора решает проблему локально, но такой подход совершенно не масштабируется, нужно принципиально другое архитектурное решение - например, механизм внимания (см следующий раздел)

У нас есть размеченый текст. Как его нарезать для обучения

**Teacher forcing** [(Williams & Zipser, 1989)](https://doi.org/10.1162/neco.1989.1.2.270)<br>Идея в следующем - учим модель генерировать ответ по одному токену - передаем на вход контекст из обучающей выборки и просим сгенерировать правильный токен. Иначе ошибки будут просто накапливаться

<img src="img/teacher_forcing.png" width=350>

Аналогия - автоинструктор проверяет навыки студента-водителя отдельно на каждом повороте. Если тот ошибается, инструктор и сразу же корректирует траекторию и они переходят к следюущему повороту. В конце разбирают все допущенные ошибки

**Exposure bias** [(Ranzato et al., 2015)](https://arxiv.org/abs/1511.06732)<br>
На инференсе никакой разметки (автоинструктора) нет, модель продолжать префиксы, которые могла не видеть при обучении. 

Бороться с этим эффектом можно переходя на обучение на уровне целых последовательностей (RL поверх метрики, beam-aware функции потерь), либо применяя более мягкий вариант scheduled sampling

**Scheduled sampling** [(Bengio et al., 2015)](https://arxiv.org/abs/1506.03099)<br>Чтобы побороть негативный эффект Exposure bias Bengio с коллегами предложил гибридный палн обучения - чередовать Teacher Forcing с обчением на сгенерированных токенах. Коэффицент замешивания - это параметр $\epsilon$ и он убывает по расписанию. Авторы таким образом реализуют классическуб для машинного обучения стратегию плавного перехода от Exploration к Exploitation. Расписание может быть линейным, экспоненциальным $\epsilon_i=k^{i}$ или обратно-сигмоидное $\epsilon_i=k/(k+e^{i/k})$

Разрыв смягчается, хотя целевая функция перестаёт быть корректным правдоподобием (оценка смещена)

## Механизм внимания

**Механизм внимания** [(Bahdanau et al., 2014)](https://arxiv.org/abs/1409.0473) был предложен, чтобы устранить горлышко: декодер на каждом шаге генерации "видит" все токены входной последовательности и их представления полученные энкодером

1) оцениваем "сходство" текушего состояния декодера $s_t$ и представления каждого токена $h_s$ из входной послежовательности<br>$e_{t,s}=\operatorname{score}(s_{t-1},h_s)$<br><br>
2) все посчитанные сходства нормируем softmax, получаем вектор весов, суммируемый в 1:<br>
$\alpha_{t,s}=\frac{\exp e_{t,s}}{\sum_{s'}\exp e_{t,s'}}$<br><br>
3) считаем взвешенную сумму всех представлений энкодера<br>$c_t=\sum_{s=1}^{S}\alpha_{t,s}\,h_s$

Такой подход называют мягким выравниванием: вместо жёсткого выбора слов, как в статистическом переводе (IBM-модели), — соотвествие заменяется дифференцируемым распределением $\alpha_t$

<img src="img/attention1.png" width=250>

В оригинальном варианте Бахданау score складывался аддитивно из текущего состояния $s$ и представления токена $h_j$: $$\operatorname{score}(s,h)=v_a^{\top}\tanh(W_a s+U_a h)$$

Позднее [(Luong et al., 2015)](https://arxiv.org/abs/1508.04025) заменили его на мультипликативный способ учета: $$\operatorname{score}(s,h) = s^{\top}h \quad \text{(скалярное произведение)}$$ или $\operatorname{score}(s,h) = s^{\top}W_a h$ (обощенное). Одно матричное умножение быстрее и проще; 

При больших размерностях модели $d$ скалярные произведения растут и softmax насыщается - отсюда позже добавили масштабирование на размерность задачи $q^{\top}k/\sqrt{d_k}$ в трансформере. Люонг также систематизировал global/local attention и input feeding — подачу $c_{t-1}$ на вход следующего шага

$c_t$ — взвешенное среднее по всей входной последовательности называют __контекстным вектором__. Он пересобирается заново под каждый шаг выхода; конкатенируется с состоянием декодера перед предсказанием. Контекст перестал быть константой и стал функцией запроса. В терминах, которые скоро станут каноническими: $s$ — query, а $h_s$ — keys и values.

Матрицу весов $A=[\alpha_{t,s}]\in\mathbb{R}^{T_{output}\times T_{input}}$ называют __Alignment матрицей__. Её удобно использовать для интерпретации: тепловая карта показывает, куда «смотрело» каждое выходное слово. Для близких языков — почти диагональ; перестановки вида прилагательное–существительное en–fr видны изломами. Равномерно размазанное или залипшее на одном токене внимание - типичный симптом недообученности либо ошибок маскирования

Внимание прокладывает от каждой потери $L_t$ к каждому состоянию энкодера $h_s$ путь длины $O(1)$ — в обход цепочки из десятков рекуррентных якобианов. Самый длинный и важный маршрут (выход → вход) спрямляется; затухание внутри самих цепочек RNN остаётся. 

Концептуально внимание — дифференцируемая адресация памяти по содержимому. Осталось заметить, что она справляется и без рекуррентной «несущей», — этот шаг сделает Transformer.

## Другие архитектуры

**Pointer Networks** [(Vinyals et al., 2015)](https://arxiv.org/abs/1506.03134)<br>Энкодер-декодерная модель, которая на каждом шаге генерирует не новый токен, а позицию из входной последовательности - то есть как бы «указывает» пальцем на токены входа (отсюда название). Главное, что не требуется наличие словаря. Поэтому типовое применение - комбинаторные задачи (наполнение рюкзака, построение выпуклой оболочки), а также задачи экстрактивной суммаризации (основанной на выделении релевантных блоков текста). Архитектурно используется LSTM, позиция выбирается либо детерминированно через argmax, либо через случайное сэмплирование

**CopyNet** [(Gu et al., 2016)](https://arxiv.org/abs/1603.06393)<br>
Разрешим модели иметь свой небольшой словарь и пусть на каждом шаге генерации она может выбрать токен из словаря или токен из входа. Как выбирается - складываем два скора, сортируем сумму по убыванию, применяем softmax для нормировки и сэмплируем. 

Вероятность генерации токена из словаря реализуется стандартно, через линейный слой $Wx+b$. Вероятность выбора токена $w_i$ моделируется линейным навесом над конкатенацией $[s_j, c_j, h_i]$, где s_j  c_j h_i Решает проблему OOV-имён и редких сущностей в суммаризации и диалоге

**Pointer-generator** [(See et al., 2017)](https://arxiv.org/abs/1704.04368):<br>
Та же идея, но две вероятности замешиваются с разными весами

- вероятность выбора токена $w$ из словаря считаем стандартно для seq2seq: $P_{\text{vocab}}(w) = \text{softmax}\left( \mathbf{W}_g \mathbf{s}_t + \mathbf{b}_g \right)$
- вероятность выбора токена из входной последовательности считаем, просто как его attention вес: $P_{\text{copy}}(w) = \sum_{j: x_j = w} \alpha_{t,j}$
- коэффициент замешивания двух сигналов считаем линейным слоем по всем доступным данным: $p_{\text{gen}} = \sigma\left( \mathbf{W}_p [\mathbf{s}_t; \mathbf{c}_t; \mathbf{e}(y_{t-1})] + \mathbf{b}_p \right)$

**Neural Turing Machine** [(Graves et al., 2014)](https://arxiv.org/abs/1410.5401): контроллер-LSTM плюс внешняя матрица памяти $M\in\mathbb{R}^{N\times M}$ с дифференцируемыми чтением $r_t=\sum_i w_t(i)\,M_t(i)$ и записью; адресация контентная (косинусная близость с softmax) и позиционная (сдвиги, интерполяция). Обучается алгоритмам копирования и сортировки по примерам вход–выход. 

Попытка создать нейросеть, снабжённую внешней памятью, к которой она обращается через дифференцируемый механизм чтения и записи. Название подчеркивает сходство с асбтрактной вычислительной машиной Тьюринга, тлько здесь всё обучается градиентным спуском

Контроллер (LSTM сеть) получает на вход и прочитанное из памяти, выдаёт выход и управляющие сигналы для головок.
Память — матрица 
Головки чтения и записи — обращаются к памяти.

Обращение к памяти дискретно, чтобы оно стало дифференцируемым, используют мягкое обращение:

**Differentiable Neural Computer** [(Graves et al., 2016)](https://doi.org/10.1038/nature20101) добавляет динамическую аллокацию и темпоральные связи между записями, решает графовые задачи. Концептуальный вклад — разделение вычислителя и памяти и адресация по содержимому: дальние предки современных retrieval-механизмов

**Stack-Augmented RNN** [(Joulin & Mikolov, 2015)](https://arxiv.org/abs/1503.01007): дифференцируемый стек с мягкими push/pop. RNN конечной точности — по существу конечный автомат; стек поднимает модель на ступень выше по иерархии Хомского, давая контекстно-свободные способности: $a^{n}b^{n}$, вложенные скобки, счётчики — то, что нужно синтаксису с неограниченной вложенностью.

### Свёрточная альтернатива

**TCN** [(Bai et al., 2018)](https://arxiv.org/abs/1803.01271) развивает каузальные дилатированные свёртки **WaveNet** [(van den Oord et al., 2016)](https://arxiv.org/abs/1609.03499): рецептивное поле растёт экспоненциально с глубиной, обучение параллельно по $T$; на ряде последовательностных бенчмарков TCN обходит LSTM и GRU. Ограничение — память жёстко ограничена рецептивным полем, а состояние для стриминга — буфер длиной в это поле, не компактный вектор.

### SSM

Параллельно с равитием исследователи из Стенфорда решили переосмыслить принцип рекуррентности, взяв за сонову SSM = state space models (подробнее см главу "Альтернативные архитектуры") и описали модель **S4** [(Gu et al., 2021)](https://arxiv.org/abs/2111.00396): линейная рекурсия $h_t=\bar A h_{t-1}+\bar B x_t$, $y_t=C h_t$ (дискретизация непрерывной state-space системы). Отсутствие нелинейности между шагами даёт двойственность: обучение — свёртка с ядром $\bar K=(C\bar B,\,C\bar A\bar B,\,C\bar A^{2}\bar B,\dots)$ через FFT, параллельно; инференс — рекуррентно за $O(1)$ на шаг. Долгая память достигается не воротами, а специальной инициализацией $A$ (HiPPO); прорыв на Long Range Arena, где трансформеры проваливались. 

**Mamba** [(Gu & Dao, 2023)](https://arxiv.org/abs/2312.00752) добавляет селективность: $\bar B$, $C$ и шаг дискретизации $\Delta$ становятся функциями входа — контентно-зависимая фильтрация, функциональный наследник ворот LSTM; свёрточная форма теряется, вместо неё аппаратно-оптимизированный параллельный scan. Линейное время, константное состояние на инференсе, качество на уровне трансформеров при сопоставимых бюджетах. 

Параллельная линия — линейное внимание как RNN [(Katharopoulos et al., 2020)](https://arxiv.org/abs/2006.16236), **RWKV** [(Peng et al., 2023)](https://arxiv.org/abs/2305.13048) и **xLSTM** [(Beck et al., 2024)](https://arxiv.org/abs/2405.04517) с экспоненциальными воротами и матричной памятью. Круг замкнулся: рекуррентное состояние и ворота вернулись в мейнстрим как ответ на квадратичность внимания.

### Место классических RNN сегодня

Нишу определяет свойство $O(1)$-состояния: стриминг с жёсткой латентностью (онлайн-распознавание речи на **RNN-T** [(Graves, 2012)](https://arxiv.org/abs/1211.3711) годами работало в продакшене), edge-устройства и микроконтроллеры, малые датасеты и короткие последовательности (LSTM — по-прежнему сильный бейзлайн против переобучающегося трансформера), временные ряды и сенсорика. Плюс дидактика: состояние, ворота, teacher forcing, exposure bias — понятия, введённые здесь, работают во всём современном стеке.

## Инженерная оптимизация

**Gradient checkpointing** [(Chen et al., 2016)](https://arxiv.org/abs/1604.06174): хранить активации только в контрольных точках (например, каждые $\sqrt{T}$ шагов), остальные пересчитывать на обратном проходе: требует примерно +33% вычислений но зато память $O(\sqrt{T})$ вместо $O(T)$

**Mixed precision** [(Micikevicius et al., 2017)](https://arxiv.org/abs/1710.03740): вычисления в fp16/bf16, мастер-копия весов в fp32, loss scaling против underflow градиентов. Специфика RNN: тысячи последовательных шагов накапливают ошибку округления, а нормы состояний гуляют широко — узкая экспонента fp16 чревата переполнениями, bf16 надёжнее; и стоит использовать фьюзнутые cuDNN-ядра, иначе выигрыш съедается запуском множества мелких ядер.



INT8-квантизация весов для edge устройств и активаций даёт около четырёхкратной экономии памяти и заметное ускорение на CPU/NPU. Тонкость рекуррентности: ошибка квантизации состояния проходит через одну и ту же ячейку многократно и накапливается по шагам — на длинных последовательностях post-training квантизация деградирует, предпочтительнее quantization-aware training; аккумуляторы держать в int32, масштабы весов — поканальные, сигмоиды и tanh — таблицами

Экспортировать с динамическими осями: `dynamic_axes={'x': {0: 'batch', 1: 'time'}}` — иначе граф зафиксирует длины, встреченные при трассировке. LSTM/GRU отображаются во фьюзнутые операторы ONNX (кастомные ячейки — в циклы Scan/Loop, заметно медленнее). Рантаймы (ONNX Runtime, TensorRT) фьюзят операции и планируют статический граф; обязательная проверка — численный паритет с исходной моделью на батчах переменной длины: главный источник расхождений — маски и packing.
